# 02 — Forecast model comparison

> **Verhaallijn**: [data](01_eda.ipynb) → forecast → [simulation](03_simulation.ipynb) → [agents](04_agents.ipynb) → [results](05_results.ipynb)

**Doel**: een eerlijke vergelijking tussen drie forecasters op exact dezelfde rijen, en een gemotiveerde keuze welk model de simulator aandrijft.

**Wat dit notebook doet**:
1. Laadt drie modellen + bouwt out-of-fold predicties via leave-one-day-out CV.
2. Vergelijkt MAE/RMSE op het gemeenschappelijke (t≥6 per zone) subset zodat alle drie modellen op identieke rijen voorspellen.
3. Plot voorspellingen vs werkelijkheid op 1 mei (feestdag — moeilijkste dag).
4. Diagnoseert waarom XGBoost ondanks de laagste MAE *niet* gekozen wordt.

**De drie modellen**:
- **Naïef** — `demand[t-1]` per zone (lag-1 baseline).
- **XGBoost** (issue 1.7) — Optuna-getunede gradient boosting met lag/rolling/zone features.
- **Transformer** (issue 1.8) — sequence-model met 6-uur input window.

**Conclusie** (zie discussie + beslissing onderaan): XGBoost wint per-rij MAE maar blijkt een **MAE-degenererende constant-0 predictor** (per-bucket MAE matched `mean(demand)` tot drie decimalen). Plot toont vlak-zero curve — operationeel useless ondanks beste metric. **Transformer** wordt gekozen als simulator-engine omdat hij shape-correcte voorspellingen geeft, ook al onderschat hij magnitudes met ~50%.

In [1]:
# Torch must be imported before pandas on Windows (vendored-lib conflict).
import torch  # noqa: F401

import os, sys, pickle
from pathlib import Path

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    os.chdir(ROOT.parent)
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

from src.models.xgb_forecast import _prepare_xy as xgb_prepare_xy, _cross_validate as xgb_cv
from src.models.transformer_forecast import _build_sequences_with_meta, cross_validate as tx_cv

FIGURES = ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
DATE_LABEL = {0: '2026-04-30 (weekday)', 1: '2026-05-01 (holiday)', 2: '2026-05-02 (weekend)'}

In [2]:
features = pd.read_parquet('data/processed/features.parquet')
with open('models/xgb_v1.pkl', 'rb') as f:
    xgb_art = pickle.load(f)
print('features:', features.shape)
print('xgb best_params:', xgb_art['best_params'])

features: (65592, 17)
xgb best_params: {'max_depth': 3, 'learning_rate': 0.07696814351094777, 'n_estimators': 101, 'subsample': 0.9490134743036945, 'colsample_bytree': 0.675760829066606}


## OOF predicties berekenen

Leave-one-day-out per model. Naïef is direct, XGBoost herbruikt de getunede hyperparameters, Transformer hertraint per fold.

In [3]:
X_xgb, y_xgb = xgb_prepare_xy(features)
folds_xgb = features['fold'].to_numpy()
xgb_oof, xgb_per_fold = xgb_cv(X_xgb, y_xgb, folds_xgb, xgb_art['best_params'])
print('xgb OOF on full features:', xgb_oof.shape)

xgb OOF on full features: (65592,)


In [4]:
X_tx, y_tx, folds_tx, meta = _build_sequences_with_meta(features)
tx_oof, tx_per_fold, _ = tx_cv(X_tx, y_tx, folds_tx)
print('transformer OOF on sequences:', tx_oof.shape)

transformer OOF on sequences: (60126,)


In [5]:
# Restrict to the sequence subset (t>=6 per zone) so all three models compete on identical rows
feat_with_xgb = features.copy()
feat_with_xgb['xgb_pred'] = xgb_oof
cmp = meta.merge(feat_with_xgb[['h3_cell', 'timestamp', 'xgb_pred', 'demand_lag_1']],
                 on=['h3_cell', 'timestamp'], how='left')
cmp['naive_pred'] = cmp['demand_lag_1'].astype(float)
cmp['tx_pred'] = tx_oof.astype(float)
cmp = cmp.rename(columns={'demand_lag_1': 'lag1'})
print('comparison rows:', len(cmp))
cmp.head(3)

comparison rows: 60126


,h3_cell,timestamp,demand,fold,xgb_pred,lag1,naive_pred,tx_pred
0,89194d86ac3ffff,2026-04-30 06:00:00,0.0,0,0.0,0,0.0,0.017110
1,89194d86ac3ffff,2026-04-30 07:00:00,0.0,0,0.0,0,0.0,0.008361
2,89194d86ac3ffff,2026-04-30 08:00:00,0.0,0,0.0,0,0.0,0.010040


## Tabel: MAE / RMSE per held-out dag

Lager is beter. Naïef is de ondergrens om verbetering te kwantificeren.

In [6]:
rows = []
for fold in [0, 1, 2]:
    sub = cmp[cmp['fold'] == fold]
    actual = sub['demand'].to_numpy()
    for name, col in [('naive', 'naive_pred'), ('xgboost', 'xgb_pred'), ('transformer', 'tx_pred')]:
        pred = sub[col].to_numpy()
        rows.append({
            'fold': fold,
            'date': DATE_LABEL[fold],
            'model': name,
            'mae': mean_absolute_error(actual, pred),
            'rmse': float(np.sqrt(mean_squared_error(actual, pred))),
        })
per_fold = pd.DataFrame(rows)
wide = per_fold.pivot(index='date', columns='model', values='mae').round(4)
wide.columns = [f'{c}_MAE' for c in wide.columns]
rmse_wide = per_fold.pivot(index='date', columns='model', values='rmse').round(3)
rmse_wide.columns = [f'{c}_RMSE' for c in rmse_wide.columns]
summary = pd.concat([wide, rmse_wide], axis=1)
summary

,naive_MAE,transformer_MAE,xgboost_MAE,naive_RMSE,transformer_RMSE,xgboost_RMSE
date,,,,,,
2026-04-30 (weekday),0.0965,0.1003,0.0597,0.630,0.471,0.483
2026-05-01 (holiday),0.1337,0.1304,0.0898,0.727,0.581,0.610
2026-05-02 (weekend),0.0748,0.0947,0.0476,0.491,0.374,0.389


In [7]:
rows = []
for name, col in [('naive', 'naive_pred'), ('xgboost', 'xgb_pred'), ('transformer', 'tx_pred')]:
    pred = cmp[col].to_numpy()
    actual = cmp['demand'].to_numpy()
    rows.append({'model': name,
                 'mae_overall': mean_absolute_error(actual, pred),
                 'rmse_overall': float(np.sqrt(mean_squared_error(actual, pred)))})
overall = pd.DataFrame(rows).set_index('model').round(4)
overall

,mae_overall,rmse_overall
model,,
naive,0.1022,0.6230
xgboost,0.0663,0.5038
transformer,0.1092,0.4839


In [8]:
zone_demand = features.groupby('h3_cell')['demand'].sum()
bucket = pd.qcut(zone_demand.rank(method='first'), q=3, labels=['low', 'mid', 'high'])
cmp['zone_bucket'] = cmp['h3_cell'].map(bucket.to_dict())
rows = []
for b in ['low', 'mid', 'high']:
    sub = cmp[cmp['zone_bucket'] == b]
    for name, col in [('naive', 'naive_pred'), ('xgboost', 'xgb_pred'), ('transformer', 'tx_pred')]:
        rows.append({
            'bucket': b, 'model': name,
            'mae': mean_absolute_error(sub['demand'], sub[col]),
            'rmse': float(np.sqrt(mean_squared_error(sub['demand'], sub[col]))),
        })
per_bucket = pd.DataFrame(rows).pivot(index='bucket', columns='model', values='mae').round(4)
per_bucket = per_bucket.reindex(['low', 'mid', 'high'])
per_bucket

model,naive,transformer,xgboost
bucket,,,
low,0.0314,0.0605,0.0160
mid,0.0620,0.0829,0.0398
high,0.2129,0.1842,0.1429


## Plot: voorspelling vs werkelijkheid op 1 mei

Aggregate per uur (som over alle zones) voor de feestdag — de moeilijkste dag om te voorspellen, en daarom het meest informatieve oordeel.

In [9]:
day = cmp[cmp['fold'] == 1].copy()
day['hour'] = day['timestamp'].dt.hour
hourly = day.groupby('hour').agg(
    actual=('demand', 'sum'),
    naive=('naive_pred', 'sum'),
    xgboost=('xgb_pred', 'sum'),
    transformer=('tx_pred', 'sum'),
).reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(hourly['hour'], hourly['actual'], color='black', linewidth=2.4, label='actual', marker='o')
ax.plot(hourly['hour'], hourly['naive'], color='#9c9c9c', linestyle=':', label='naive (lag-1)')
ax.plot(hourly['hour'], hourly['xgboost'], color='#1f77b4', label='XGBoost')
ax.plot(hourly['hour'], hourly['transformer'], color='#d62728', label='Transformer')
ax.set_xlabel('Uur (UTC)')
ax.set_ylabel('Demand (som over alle zones)')
ax.set_title('Held-out dag 1 mei (Dag van de Arbeid) — voorspelling vs werkelijkheid')
ax.set_xticks(range(0, 24, 2))
ax.grid(True, alpha=0.3)
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES / 'forecast_comparison_1mei.png', dpi=110, bbox_inches='tight')
plt.show()

C:\Users\ralph\AppData\Local\Temp\ipykernel_34804\1783066763.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Discussie

**Wat de cijfers eerst lijken te zeggen:**

- **XGBoost wint per-rij MAE op alle slices.** Per-fold (0.060 / 0.090 / 0.048) en per zone-bucket (0.016 / 0.040 / 0.143).
- **Transformer is per-rij MAE iets slechter dan zelfs naïef** (0.109 vs 0.102 overall) — de "Transformer onder-presteert door beperkte data"-intuïtie uit de issue lijkt bevestigd.

**Maar de plot vertelt een ander verhaal:**

Op de held-out dag 1 mei (boven) zie je:
- Actual piekt op 270 om 13u.
- Naïef (gestippeld grijs) reproduceert de piek-vorm met ~1 uur vertraging — magnitude correct.
- Transformer (rood) ondershoot maar volgt de vorm: piek op ~150 om 13u, juiste timing.
- **XGBoost (blauw) blijft de hele dag vlak rond 0.** Geen piek. Helemaal niets.

**Waarom is XGBoost vlak?**

We tunden XGBoost met `objective=reg:absoluteerror` (MAE-loss). Op een target die voor 96% uit nullen bestaat is de MAE-optimale puntpredictie de **mediaan**, en die mediaan is **0**. XGBoost heeft dus geleerd: voorspel altijd ~0 — en dat is wiskundig optimaal voor MAE op deze data.

Verificatie: XGBoost's per-bucket MAE klopt **precies** met `mean(demand)` per bucket — een model dat altijd 0 predict heeft per definitie `MAE = mean(|y|) = mean(y)` voor non-negatieve y.

| Bucket | mean(demand) | XGBoost MAE |
|---|---:|---:|
| low | 321/20064 = 0.0160 | 0.0160 |
| mid | 796/19998 = 0.0398 | 0.0398 |
| high | 2868/20064 = 0.1429 | 0.1429 |

Drie cijfers achter de komma, drie identieke matches. XGBoost is een MAE-degenererende constant-0 predictor.

**Wat dit voor onze simulator betekent:**

Per-rij MAE is de **verkeerde metric** voor sparse demand-forecasting. Een model dat altijd 0 voorspelt heeft de laagste MAE maar is operationeel waardeloos: de simulator zou nooit een rij bedienen omdat het model nergens vraag voorspelt. We hebben een model nodig dat patronen reproduceert, ook al is per-rij MAE iets hoger.

**Wanneer zou wát winnen?**

- **XGBoost-MAE wint per-row metric** maar predict effectief de mediaan op sparse data → operationeel onbruikbaar zonder objective-fix.
- **Transformer wint shape-reproductie** ondanks hogere per-row MAE; produceert non-triviale predicties met juiste timing.
- **Naïef (lag-1) wint shape én magnitude** met ~1u delay, maar vereist tijdens prediction de daadwerkelijke demand van het vorige uur — alleen bruikbaar voor nowcasting, niet voor forecasting van morgen.
- **XGBoost zou competitief kunnen worden** met `objective=reg:squarederror` (predict mean ipv mediaan) of met log-transformatie van de target. Dat is een aanpassing voor de modeling-issue, niet voor deze vergelijking.

## Beslissing — Transformer (`models/transformer_v1.pt`) drijft de simulator aan

**Waarom niet XGBoost, ondanks de laagste MAE?** Omdat het een MAE-degenererende constant-0 predictor blijkt te zijn (zie discussie). Per-rij MAE = mean(demand) per bucket — drie decimalen overeenkomst, geen toeval. Voor de simulator levert XGBoost geen bruikbare per-zone-uur voorspellingen.

**Waarom niet naïef?** Naïef vereist `demand[t-1]` op het moment van predictie, wat alleen beschikbaar is voor nowcasting. De simulator moet meerdere uren vooruit kunnen voorspellen zonder die ground-truth.

**Waarom Transformer:**

1. **Geeft niet-triviale, shape-correcte voorspellingen.** Op 1 mei volgt de Transformer de juiste daily curve (piek om 13u, dal om 15u, tweede piek om 18u) — magnitudes onderschat (~50%), maar de *vorm* die een simulator nodig heeft is er.
2. **Per-bucket MAE niet bezwaarlijk.** Op high-bucket: 0.184 vs XGBoost 0.143, maar XGBoost-0.143 = puur de mean. Transformer-0.184 doet daar bovenop iets nuttigs (varieert per zone/uur).
3. **Schaalt mee bij meer data.** Wanneer de productie-export uitbreidt naar meerdere weken, leveren sequence-modellen meer winst dan boosted trees omdat ze wekelijkse patronen kunnen leren.

**Caveats die voor downstream issues open blijven:**

- Magnitude-onderschatting (~50% op piek) → simulator kan een lineaire kalibratie nodig hebben, of een log-target retrain.
- Per-rij MAE is niet de juiste validatie-metric voor sparse demand. Voor de modeling-issue moet er een metric komen die shape-reproductie meet (bv. dynamic time warping op uur-aggregaten, of sum-correlation per dag).
- XGBoost-as-baseline kan opnieuw competitief worden met `objective=reg:squarederror` of log-target — zie [docs/limitations.md](../docs/limitations.md).

---

**Volgende**: [03_simulation.ipynb](03_simulation.ipynb) — valideert of de simulator (gevoed door deze Transformer) werkelijke dag-totalen kan reproduceren via replay-test.